In [1]:
import json
import os
import requests

def download_and_load_file(url, local_path):
    if not os.path.exists(local_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text = response.text
        with open(local_path, "w", encoding="utf-8") as file:
            file.write(text)
    else:
        with open(local_path, "r", encoding="utf-8") as file:
            text = file.read()

    data = json.loads(text)
    return data

In [2]:
file_path = "instruction-data-with-preference.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/04_preference-tuning-with-dpo/instruction-data-with-preference.json"
)

data = download_and_load_file(url, file_path)
print(len(data))

1100


In [3]:
import pprint
pprint.pp(data[0])

{'instruction': 'Evaluate the following phrase by transforming it into the '
                'spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the '
           'correct spelling is "friend".',
 'rejected': 'The spelling of the given phrase "freind" is flat out wrong, get '
             'it together, the correct spelling is "friend".',
 'chosen': 'The spelling of the given phrase "freind" is incorrect, the '
           'correct spelling is "friend".'}


In [4]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (f"\n\n### Input:\n{entry['input']}" if entry["input"] else "")
    return instruction_text + input_text

In [5]:
print(format_input(data[5]))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Suggest a more formal synonym for "happy."


In [31]:
import torch
from torch.utils.data import Dataset

class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        super().__init__()
        self.data = data
        self.encoded_texts = []

        for entry in self.data:
            prompt = format_input(entry)
            chosen_text = entry["chosen"]
            rejected_text = entry["rejected"]

            prompt_tokens = tokenizer.encode(prompt)
            choosen_full_text = f"{prompt}\n\n### Response:\n{chosen_text}"
            rejected_full_text = f"{prompt}\n\n### Response:\n{rejected_text}"
            choosen_full_tokens = tokenizer.encode(choosen_full_text)
            rejected_full_tokens = tokenizer.encode(rejected_full_text)

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": choosen_full_tokens,
                "rejected": rejected_full_tokens
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [32]:
def custom_collate_fn(batch, pad_token_id=50256, allowed_max_length=None, mask_prompt_tokens=True, device="cpu"):

    batch_data = {
        "prompt":[],
        "chosen":[],
        "rejected":[],
        "chosen_mask":[],
        "rejected_mask":[]
    }

    max_length = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max_length = max(len(entry[key])+1 for entry in batch)
            max_length = max(max_length, current_max_length)


    for item in batch:
        batch_data["prompt"].append(torch.tensor(item["prompt"]))
        for key in ["chosen", "rejected"]:
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length - len(sequence))

            mask = torch.ones(len(padded)).bool()
            mask[len(sequence):] = False

            if mask_prompt_tokens:
                # +2 to remove \n\n before ### Response:
                mask[:len(item["prompt"]) + 2] = False
            
            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        tensor_stack = torch.stack(batch_data[key])
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]
        batch_data[key] = tensor_stack.to(device)
    
    return batch_data

In [33]:
from functools import partial

device = "cuda" if torch.cuda.is_available() else "cpu"

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    mask_prompt_tokens=True,
    allowed_max_length=1024
)

In [48]:
# testing customized collate function
import tiktoken
from torch.utils.data import DataLoader

tokenizer = tiktoken.get_encoding("gpt2")
example_data = data[:2]

dataset = PreferenceDataset(example_data, tokenizer)
dataloader = DataLoader(
    dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False,
)


In [49]:
for batch in dataloader:
    pass

print(batch.keys())

dict_keys(['prompt', 'chosen', 'rejected', 'chosen_mask', 'rejected_mask'])


In [50]:
batch['prompt']

[tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,    36,  2100,  4985,   262,  1708,  9546,
           416, 25449,   340,   656,   262, 24993,  1813,    13,   198,   198,
         21017, 23412,    25,   198, 19503,   521, 14610,  1545]),
 tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198, 18378,   262,  1708,  6827,   329, 23491,
            13,   198,   198, 21017, 23412,    25,   198,  1544,   467,   284,
           262,  3952,   790,  1110,    13])]

In [53]:
batch['chosen']

tensor([[21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,    36,  2100,  4985,   262,  1708,  9546,
           416, 25449,   340,   656,   262, 24993,  1813,    13,   198,   198,
         21017, 23412,    25,   198, 19503,   521, 14610,  1545,   198,   198,
         21017, 18261,    25,   198,   464, 24993,   286,   262,  1813,  9546,
           366, 19503,   521,     1,   318, 11491,    11,   262,  3376, 24993,
           318,   366,  6726,  1911, 50256, 50256, 50256, 50256, 50256, 50256,
         50256],
        [21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198, 18378,   262,  1708,  6827,   329, 23491,
            13,   198,   198, 21017, 23412,    25,   198,  1544,   467,   284,
           262,  3952,   790,  1110

In [54]:
def decode_tokens_from_batch(token_ids, tokenizer):
    tokens_list = token_ids.flatten().tolist()
    return tokenizer.decode(tokens_list)

In [55]:
text = decode_tokens_from_batch(
    token_ids=batch['prompt'][0],
    tokenizer=tokenizer
)
print(text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend


In [56]:
text = decode_tokens_from_batch(
    token_ids=batch['chosen'][0],
    tokenizer=tokenizer
)
print(text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [57]:
print(batch['chosen_mask'])

tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True, False, False, False, False, False, False,
         False],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False

In [61]:
text = decode_tokens_from_batch(
    token_ids=batch['chosen'][0][batch['chosen_mask'][0]],
    tokenizer=tokenizer
)
print(text)

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".


In [63]:
# creating training, testing and validation data loaders
# training : testing : validation => 85 : 10 : 5
train_portion = int(.85 * len(data))
test_portion = int(.10 * len(data))
val_portion = len(data) - test_portion - train_portion

train_data = data[:train_portion]
test_data = data[train_portion : train_portion + test_portion]
val_data = data[train_portion + test_portion : ]